# DUKASCOPY

### IMPORTS

In [1]:
!pip install dukascopy-python pandas pyarrow joblib -q

import os
import time
import pandas as pd
import dukascopy_python as dp
from datetime import datetime
from pathlib import Path
from joblib import Parallel, delayed

### PARAMS

In [2]:
N_JOBS = 4 

PAIRS = ["USD/ZAR"]

MONTHS = [
    "202401"
]

### GOOGLE DRIVE

In [3]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = Path("/content/drive/MyDrive/GITHUB-COPILOT/stk-mat2011/data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


### GET MONTH RANGE

In [4]:
def get_month_range(yyyymm: str):
    year, month = int(yyyymm[:4]), int(yyyymm[4:6])
    start = datetime(year, month, 1)
    end = datetime(year + 1, 1, 1) if month == 12 else datetime(year, month + 1, 1)
    return start, end

### PROCESS JOB

In [5]:
def process_job(pair, yyyymm, compression="snappy", max_retries=3):
    """Function to be executed in parallel with existence check and retries"""
    symbol = pair.replace("/", "").lower()
    
    # 1. EXISTENCE CHECK: Define file paths first
    out_name_bid = f"{symbol}_dukascopy_bid_{yyyymm}.parquet"
    out_name_ask = f"{symbol}_dukascopy_ask_{yyyymm}.parquet"
    out_path_bid = OUT_DIR / out_name_bid
    out_path_ask = OUT_DIR / out_name_ask

    # If both files already exist, skip the download completely
    if out_path_bid.exists() and out_path_ask.exists():
        return f"⏭️ Skipped (Already Exists): {pair} {yyyymm}"

    start, end = get_month_range(yyyymm)
    
    # 2. RETRY LOGIC: Handle Dukascopy connection drops
    for attempt in range(max_retries):
        try:
            df = dp.fetch(
                instrument=pair,
                interval=dp.INTERVAL_TICK,
                offer_side=dp.OFFER_SIDE_BID, 
                start=start,
                end=end,
            )

            if df is None or len(df) == 0:
                return f"⚠️ Empty: {pair} {yyyymm}"

            results_stats = []
            for side, price_col, vol_col in [("bid", "bidPrice", "bidVolume"), ("ask", "askPrice", "askVolume")]:
                out_name = out_name_bid if side == "bid" else out_name_ask
                out_path = out_path_bid if side == "bid" else out_path_ask
                
                pd.DataFrame({
                    "datetime": df.index,
                    "symbol": symbol.upper(),
                    "price_type": side.upper(),
                    "price": df[price_col].values,
                    "volume": df[vol_col].values,
                }).to_parquet(out_path, engine="pyarrow", compression=compression, index=False)
                
                results_stats.append(f"{out_name} ({len(df):,} rows)")
                
            return f"✅ Success: {pair} {yyyymm} -> " + " & ".join(results_stats)
        
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(5) # Wait 5 seconds before retrying
                continue
            return f"❌ FAILED: {pair} {yyyymm} | Error after {max_retries} attempts: {str(e)}"


### EXECUTION

In [7]:
jobs = [(p, m) for p in PAIRS for m in MONTHS]
print(f"Downloading {len(jobs)} jobs · {N_JOBS} workers\n")

t0 = time.time()
results = Parallel(n_jobs=N_JOBS, backend="threading")(
    delayed(process_job)(p, m) for p, m in jobs
)
for r in results:
    print(f"  {r}")

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s")
print(f"Saved to {OUT_DIR.absolute()}")

INFO:DUKASCRIPT:current timestamp :2024-01-18T09:47:32.112000
INFO:DUKASCRIPT:current timestamp :2024-01-02T09:06:44.320000
INFO:DUKASCRIPT:current timestamp :2024-01-18T14:09:20.151000
INFO:DUKASCRIPT:current timestamp :2024-01-02T13:44:39.071000
INFO:DUKASCRIPT:current timestamp :2024-01-18T17:46:07.539000
INFO:DUKASCRIPT:current timestamp :2024-01-02T18:05:26.885000
INFO:DUKASCRIPT:current timestamp :2024-01-19T05:41:49.601000
INFO:DUKASCRIPT:current timestamp :2024-01-03T03:16:37.444000
INFO:DUKASCRIPT:current timestamp :2024-01-19T09:46:13.395000
INFO:DUKASCRIPT:current timestamp :2024-01-03T10:01:04.690000
INFO:DUKASCRIPT:current timestamp :2024-01-19T14:41:04.526000
INFO:DUKASCRIPT:current timestamp :2024-01-03T14:08:46.868000
INFO:DUKASCRIPT:current timestamp :2024-01-19T18:49:11.649000
INFO:DUKASCRIPT:current timestamp :2024-01-03T17:59:52.182000
INFO:DUKASCRIPT:current timestamp :2024-01-22T05:14:36.574000
INFO:DUKASCRIPT:current timestamp :2024-01-03T21:52:44.496000
INFO:DUK

  ✅ Success: USD/ZAR 202401 -> usdzar_dukascopy_bid_202401.parquet (2,504,167 rows) & usdzar_dukascopy_ask_202401.parquet (2,504,167 rows)

Done in 120.6s
Saved to /content/drive/MyDrive/GITHUB-COPILOT/stk-mat2011/data/processed
